# PG-Stack · Step 5c — Validating the label-adaptive prior-anchored blend

**Method (fixed BEFORE this run):**  score = w · rank(U) + (1 − w) · rank(M3),  with  **w = k / (k + n),  k = 6**,
where *n* = number of labelled cancers in the training set, U = zero-label graph ugly-duckling score, M3 = LightGBM with global + graph-prior features.
Ranks are taken within each test set. Nothing in this notebook tunes k.

**Pre-registered claims**
1. The adaptive blend is **never significantly worse** than the better of {M3, U} at any label budget.
2. It is **significantly better than M3** when labels are scarce and **significantly better than U** once a few labels exist.

**C1 — fresh within-dataset learning curve:** new seeds and new training subsets (1, 2, 3, 5, 10, 20 % of patients, 10 repeats).
**C2 — low-label + unseen hospital:** train on a small share of patients from the *other* hospitals, test on the held-out hospital (6 hospitals with ≥ 10 cancers, 5 repeats). This setting was never used to choose k.

CPU is enough. Runtime ≈ 40–60 min. Resumable.

In [ ]:
import os, sys, glob, time, gc, warnings
import numpy as np, pandas as pd, lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")

WORK = "/kaggle/working"
COMP_DIR = "/kaggle/input/competitions/isic-2024-challenge"
SEED = 42                      # feature construction (same as before)
SEED_C = 2026                  # NEW seed for all training subsets in this notebook
K_PRIOR = 6                    # fixed in advance - do not change after seeing results
FRACTIONS_C1 = [0.01, 0.02, 0.03, 0.05, 0.10, 0.20]
REPEATS_C1 = 10
FRACTIONS_C2 = [0.01, 0.02, 0.05, 0.10]
REPEATS_C2 = 5
N_BAGS = 3
NEG_PER_POS = 20
N_BOOT = 100
MIN_POS_HOSPITAL = 10

def locate(name):
    if os.path.exists(f"{WORK}/{name}"): return f"{WORK}/{name}"
    for d in range(1, 5):
        hits = glob.glob("/kaggle/input/" + "*/" * d + name)
        if hits: return hits[0]
    return None
paths = {n: locate(n) for n in ["oof_step3.csv", "emb_pca32.npy", "pgstack_eval.py"]}
assert all(paths.values()), f"missing: {[k for k, v in paths.items() if v is None]}"
sys.path.insert(0, os.path.dirname(paths["pgstack_eval.py"]))
from pgstack_eval import *
print(paths, "| k =", K_PRIOR)

## 1–3. Features (identical to Steps 5 / 5b)

In [ ]:
oof = pd.read_csv(paths["oof_step3.csv"])
df = pd.read_csv(f"{COMP_DIR}/train-metadata.csv", low_memory=False)
df = oof[["isic_id", "fold"]].merge(df, on="isic_id", how="left")
assert (df.isic_id.values == oof.isic_id.values).all()
emb = np.load(paths["emb_pca32.npy"]).astype(np.float32)
assert len(emb) == len(oof)

LEAK = ["lesion_id", "iddx_full", "iddx_1", "iddx_2", "iddx_3", "iddx_4", "iddx_5",
        "mel_mitotic_index", "mel_thick_mm", "tbp_lv_dnn_lesion_confidence"]
NUM_RAW = [c for c in df.columns if (c.startswith("tbp_lv_") or c in ["age_approx", "clin_size_long_diam_mm"])
           and c not in LEAK and pd.api.types.is_numeric_dtype(df[c])]
CAT_COLS = ["sex", "anatom_site_general", "tbp_tile_type", "tbp_lv_location", "tbp_lv_location_simple"]

def add_lesion_features(d):
    d = d.copy(); eps = 1e-6
    d["age_approx"] = pd.to_numeric(d["age_approx"], errors="coerce")
    d["lesion_size_ratio"] = d.tbp_lv_minorAxisMM / (d.clin_size_long_diam_mm + eps)
    d["lesion_shape_index"] = d.tbp_lv_areaMM2 / (d.tbp_lv_perimeterMM ** 2 + eps)
    d["hue_contrast"] = (d.tbp_lv_H - d.tbp_lv_Hext).abs()
    d["luminance_contrast"] = (d.tbp_lv_L - d.tbp_lv_Lext).abs()
    d["lesion_color_difference"] = np.sqrt(d.tbp_lv_deltaA**2 + d.tbp_lv_deltaB**2 + d.tbp_lv_deltaL**2)
    d["border_complexity"] = d.tbp_lv_norm_border + d.tbp_lv_symm_2axis
    d["color_uniformity"] = d.tbp_lv_color_std_mean / (d.tbp_lv_radial_color_std_max + eps)
    d["position_distance_3d"] = np.sqrt(d.tbp_lv_x**2 + d.tbp_lv_y**2 + d.tbp_lv_z**2)
    d["perimeter_to_area_ratio"] = d.tbp_lv_perimeterMM / (d.tbp_lv_areaMM2 + eps)
    d["area_to_perimeter_ratio"] = d.tbp_lv_areaMM2 / (d.tbp_lv_perimeterMM + eps)
    d["lesion_visibility_score"] = d.tbp_lv_deltaLBnorm + d.tbp_lv_norm_color
    d["symmetry_border_consistency"] = d.tbp_lv_symm_2axis * d.tbp_lv_norm_border
    d["color_shape_composite"] = (d.tbp_lv_color_std_mean + d.tbp_lv_area_perim_ratio + d.tbp_lv_symm_2axis) / 3
    d["log_area"] = np.log1p(d.tbp_lv_areaMM2)
    d["normalized_lesion_size"] = d.clin_size_long_diam_mm / (d.age_approx + eps)
    d["mean_hue_difference"] = (d.tbp_lv_H + d.tbp_lv_Hext) / 2
    d["std_dev_contrast"] = np.sqrt((d.tbp_lv_deltaA**2 + d.tbp_lv_deltaB**2 + d.tbp_lv_deltaL**2) / 3)
    return d

ENG = ["lesion_size_ratio", "lesion_shape_index", "hue_contrast", "luminance_contrast", "lesion_color_difference",
       "border_complexity", "color_uniformity", "position_distance_3d", "perimeter_to_area_ratio",
       "area_to_perimeter_ratio", "lesion_visibility_score", "symmetry_border_consistency",
       "color_shape_composite", "log_area", "normalized_lesion_size", "mean_hue_difference", "std_dev_contrast"]
df = add_lesion_features(df)
LESION_NUM = sorted(set(NUM_RAW + ENG))
for c in CAT_COLS: df[c] = df[c].fillna("NA").astype("category")
y, folds, pid = oof.target.values, oof.fold.values, oof.patient_id.values
N_FOLDS = int(folds.max()) + 1
POS_COLS = ["tbp_lv_x", "tbp_lv_y", "tbp_lv_z", "position_distance_3d"]
TAB_COLS = [c for c in LESION_NUM if c not in POS_COLS]        # appearance/shape only, for the tabular graph
print(len(df), "lesions |", len(LESION_NUM), "lesion features")

In [ ]:
df["hospital"] = df["attribution"].fillna("unknown").astype(str)
print(df.groupby("hospital").agg(lesions=("isic_id", "size"), cancers=("target", "sum"),
                                 patients=("patient_id", "nunique")).sort_values("cancers", ascending=False))

In [ ]:
g = df.groupby("patient_id"); GLOBAL = []
for c in LESION_NUM:
    df[f"{c}_pnorm"] = (df[c] - g[c].transform("mean")) / (g[c].transform("std") + 1e-6); GLOBAL.append(f"{c}_pnorm")
df["patient_n_lesions"] = g["isic_id"].transform("size"); GLOBAL.append("patient_n_lesions")

In [ ]:
X = df[LESION_NUM].astype(np.float32); X = X.fillna(X.median()).values
tab_idx = [LESION_NUM.index(c) for c in TAB_COLS]
TAB = StandardScaler().fit_transform(X[:, tab_idx]).clip(-6, 6).astype(np.float32)
TAB10 = PCA(10, random_state=SEED).fit_transform(TAB).astype(np.float32)
XYZ = df[["tbp_lv_x", "tbp_lv_y", "tbp_lv_z"]].fillna(0).values.astype(np.float32)
SP = {"tab": TAB, "app": emb, "pos": XYZ}
n = len(df)
GP = {f"gp_{s}_k{k}_{t}": np.full(n, np.nan, np.float32) for s in SP for k in [5, 10, 20] for t in ["dist", "lof"]}
GP["gp_mahal"] = np.full(n, np.nan, np.float32); GP["gp_iforest"] = np.full(n, np.nan, np.float32)

def lof(dist, idx, k):
    d, i = dist[:, :k], idx[:, :k]; reach = np.maximum(d, d[:, -1][i])
    lrd = 1.0 / (reach.mean(1) + 1e-6); return lrd[i].mean(1) / lrd

t0 = time.time()
for j, (p, rows) in enumerate(pd.Series(np.arange(n)).groupby(pid).indices.items()):
    r = np.asarray(rows); m = len(r)
    if m < 6: continue
    km = min(20, m - 1)
    for s, Z in SP.items():
        dist, idx = NearestNeighbors(n_neighbors=km + 1).fit(Z[r]).kneighbors(Z[r]); dist, idx = dist[:, 1:], idx[:, 1:]
        for k in [5, 10, 20]:
            kk = min(k, km); GP[f"gp_{s}_k{k}_dist"][r] = dist[:, :kk].mean(1); GP[f"gp_{s}_k{k}_lof"][r] = lof(dist, idx, kk)
    if m >= 15:
        Zp = TAB10[r]; dz = Zp - Zp.mean(0); C = np.cov(Zp, rowvar=False) + 1e-3 * np.eye(10)
        GP["gp_mahal"][r] = np.sqrt(np.einsum("ij,jk,ik->i", dz, np.linalg.inv(C), dz))
    if m >= 10:
        Zi = np.c_[TAB10[r], emb[r, :10]]
        GP["gp_iforest"][r] = -IsolationForest(n_estimators=100, random_state=SEED).fit(Zi).score_samples(Zi)
    if j % 200 == 0: print(f"  patient {j} | {time.time()-t0:.0f}s")
for k, v in GP.items(): df[k] = v
key = df.patient_id.astype(str) + "|" + df.anatom_site_general.astype(str)
df["gp_pos_k10_dist_siteadj"] = df["gp_pos_k10_dist"] / (df.groupby(key)["gp_pos_k10_dist"].transform("median") + 1e-6)
PRIOR = list(GP.keys()) + ["gp_pos_k10_dist_siteadj"]
print(f"graph prior: {len(PRIOR)} features in {time.time()-t0:.0f}s")

# zero-label score: average percentile rank of the 3 strongest label-free scores
U = np.nanmean(np.stack([pd.Series(df[c]).rank(pct=True).values for c in ["gp_tab_k10_lof", "gp_mahal", "gp_iforest"]]), 0)
U = np.nan_to_num(U, nan=0.5)
print("zero-label U score: AUROC", round(roc_auc_score(y, U), 4), "| pAUC80", round(pauc_80(y, U), 4))

## 4. Trainer (identical to Step 5b)

In [ ]:
BASE_F = LESION_NUM + CAT_COLS
FSETS = {"M0_lesion": BASE_F, "M1_global": BASE_F + GLOBAL,
         "M2_prior": BASE_F + PRIOR, "M3_global_prior": BASE_F + GLOBAL + PRIOR}
MODELS = list(FSETS)
XALL = {m: df[f] for m, f in FSETS.items()}
BASE_PARAMS = dict(objective="binary", learning_rate=0.03, num_leaves=63, max_depth=7,
                   feature_fraction=0.6, bagging_fraction=0.8, bagging_freq=1, lambda_l1=0.1, lambda_l2=1.0,
                   n_estimators=300, verbosity=-1)

def fit_predict(model, tr_rows, te_rows, seed):
    pos, neg = tr_rows[y[tr_rows] == 1], tr_rows[y[tr_rows] == 0]
    Xte = XALL[model].iloc[te_rows]; pred = np.zeros(len(te_rows))
    for b in range(N_BAGS):
        rng = np.random.default_rng(seed + b)
        sub = np.r_[pos, rng.choice(neg, min(len(neg), NEG_PER_POS * len(pos)), replace=False)]
        mcs = int(np.clip(len(sub) / 50, 5, 40))
        m = lgb.LGBMClassifier(**BASE_PARAMS, min_child_samples=mcs, random_state=seed + b).fit(XALL[model].iloc[sub], y[sub])
        pred += m.predict_proba(Xte)[:, 1] / N_BAGS
    return pred

pat = df.groupby("patient_id").agg(fold=("fold", "first"), has_pos=("target", "max"),
                                   hospital=("hospital", lambda s: s.mode().iat[0]))
rows_of = pd.Series(np.arange(n)).groupby(pid).indices

def sample_patients(ids_df, share, rng):
    chosen = []
    for hp in [0, 1]:
        ids = ids_df.index[ids_df.has_pos == hp].values
        k = int(round(share * len(ids)))
        if hp == 1: k = max(1, k)                      # at least one cancer patient
        if k > 0: chosen += list(rng.choice(ids, min(k, len(ids)), replace=False))
    return chosen

def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.zeros(len(p)); run = 0
    for rank, i in enumerate(o): run = max(run, min(1.0, (len(p) - rank) * p[i])); adj[i] = run
    return adj

In [ ]:
def rank_in(a, groups):
    # percentile rank of a within each group (e.g. each test fold)
    return pd.Series(a).groupby(groups).rank(pct=True).values

def adaptive_w(n_cancers, k=K_PRIOR): return k / (k + n_cancers)
def dersimonian_laird(est, var):
    est, var = np.asarray(est), np.asarray(var); w = 1 / var
    fixed = (w * est).sum() / w.sum(); Q = (w * (est - fixed) ** 2).sum(); dfq = len(est) - 1
    tau2 = max(0.0, (Q - dfq) / (w.sum() - (w ** 2).sum() / w.sum()))
    ws = 1 / (var + tau2); mu = (ws * est).sum() / ws.sum(); se = np.sqrt(1 / ws.sum())
    I2 = max(0.0, (Q - dfq) / Q) if Q > 0 else 0.0
    from scipy.stats import norm
    return mu, mu - 1.96 * se, mu + 1.96 * se, 1 - norm.cdf(mu / se), I2

## 5. C1 — fresh within-dataset learning curve (only M3 is trained; U needs no training)

In [ ]:
C1_PATH = f"{WORK}/step5c_c1.npz"
C1 = dict(np.load(C1_PATH)) if os.path.exists(C1_PATH) else {}
t0 = time.time()
for q in FRACTIONS_C1:
    for rep in range(REPEATS_C1):
        if f"M3|{q}|{rep}" in C1: continue
        pred = np.zeros(n, np.float32); npos = np.zeros(N_FOLDS)
        for f in range(N_FOLDS):
            rng = np.random.default_rng(SEED_C + 7919 * rep + 101 * f + int(q * 10000))
            chosen = sample_patients(pat[pat.fold != f], q, rng)
            tr = np.concatenate([rows_of[p] for p in chosen]); te = np.where(folds == f)[0]
            npos[f] = y[tr].sum()
            pred[te] = fit_predict("M3_global_prior", tr, te, SEED_C + 13 * f + 1000 * rep)
        C1[f"M3|{q}|{rep}"] = pred; C1[f"npos|{q}|{rep}"] = npos
        np.savez_compressed(C1_PATH, **C1)
    print(f"q={q:.2f}: cancers/fold={np.mean([C1[f'npos|{q}|{r}'].mean() for r in range(REPEATS_C1)]):.1f}  ({(time.time()-t0)/60:.1f} min)")

rU_fold = rank_in(U, folds)
def c1_scores(q, rep):
    m3 = C1[f"M3|{q}|{rep}"]; npos = C1[f"npos|{q}|{rep}"]
    w = adaptive_w(npos)[folds]                                    # each fold uses its own n
    rM3 = rank_in(m3, folds)
    return {"M3": m3, "U": U, "fixed_0.5": 0.5 * rU_fold + 0.5 * rM3, "adaptive": w * rU_fold + (1 - w) * rM3}
METHODS = ["M3", "U", "fixed_0.5", "adaptive"]

rows = []
for q in FRACTIONS_C1:
    S = {m: [] for m in METHODS}
    for rep in range(REPEATS_C1):
        sc = c1_scores(q, rep)
        for m in METHODS: S[m].append(pauc_80(y, sc[m]))
    npos = np.mean([C1[f"npos|{q}|{r}"].mean() for r in range(REPEATS_C1)])
    r = dict(fraction=q, cancers_per_fold=round(npos, 1), w=round(adaptive_w(npos), 3))
    r.update({m: np.mean(S[m]) for m in METHODS}); r["best_of_M3_U"] = max(r["M3"], r["U"])
    rows.append(r)
c1 = pd.DataFrame(rows); c1["adaptive_minus_best"] = c1.adaptive - c1.best_of_M3_U
print(c1.round(4).to_string(index=False))

In [ ]:
t0 = time.time()
boot = PatientBootstrap(pid, n_boot=N_BOOT, seed=SEED_C)
picks = [i for i in boot.samples if y[i].sum() > 0]
SU = np.array([pauc_80(y[i], U[i]) for i in picks])
BS = {}
for q in FRACTIONS_C1:
    per_rep = [c1_scores(q, r) for r in range(REPEATS_C1)]
    for m in ["M3", "fixed_0.5", "adaptive"]:
        BS[(m, q)] = np.array([[pauc_80(y[i], per_rep[r][m][i]) for r in range(REPEATS_C1)] for i in picks]).mean(1)
    BS[("U", q)] = SU
print(f"bootstrap {time.time()-t0:.0f}s")

out = []
for base in ["M3", "U", "fixed_0.5"]:
    block = []
    for q in FRACTIONS_C1:
        d = BS[("adaptive", q)] - BS[(base, q)]
        reps = np.array([pauc_80(y, c1_scores(q, r)["adaptive"]) - pauc_80(y, c1_scores(q, r)[base]) for r in range(REPEATS_C1)])
        block.append(dict(comparison=f"adaptive vs {base}", fraction=q, delta=round(d.mean(), 4),
                          ci95=f"[{np.percentile(d, 2.5):+.4f}, {np.percentile(d, 97.5):+.4f}]",
                          p_better=(d <= 0).mean(), p_worse=(d >= 0).mean(), repeats_positive=f"{(reps > 0).sum()}/{REPEATS_C1}"))
    for r, pa in zip(block, holm([r["p_better"] for r in block])): r["p_holm_better"] = round(pa, 4); r["sig_better"] = pa < 0.05
    for r, pa in zip(block, holm([r["p_worse"] for r in block])): r["sig_worse"] = pa < 0.05
    out += block
c1_tests = pd.DataFrame(out).drop(columns=["p_better", "p_worse"])
print(c1_tests.to_string(index=False))
print("\nClaim 1 holds if 'sig_worse' is False for every row vs M3 and vs U.")

## 6. C2 — few labelled patients from OTHER hospitals → unseen hospital

In [ ]:
HOSP = df.hospital.values
hosp_pos = df.groupby("hospital").target.sum()
HOSPS = sorted(hosp_pos[hosp_pos >= MIN_POS_HOSPITAL].index)
C2_PATH = f"{WORK}/step5c_c2.npz"
C2 = dict(np.load(C2_PATH)) if os.path.exists(C2_PATH) else {}
t0 = time.time()
for h in HOSPS:
    te = np.where(HOSP == h)[0]
    src = pat[pat.hospital != h]
    for q in FRACTIONS_C2:
        for rep in range(REPEATS_C2):
            if f"M3|{h}|{q}|{rep}" in C2: continue
            rng = np.random.default_rng(SEED_C + 31 * rep + 1009 * HOSPS.index(h) + int(q * 10000))
            chosen = sample_patients(src, q, rng)
            tr = np.concatenate([rows_of[p] for p in chosen])
            C2[f"M3|{h}|{q}|{rep}"] = fit_predict("M3_global_prior", tr, te, SEED_C + 77 * rep).astype(np.float32)
            C2[f"npos|{h}|{q}|{rep}"] = np.array([y[tr].sum()])
        np.savez_compressed(C2_PATH, **C2)
    print(f"{h[:40]:40s} test cancers={int(y[te].sum()):3d}  ({(time.time()-t0)/60:.1f} min)")

def c2_scores(h, q, rep):
    te = np.where(HOSP == h)[0]; m3 = C2[f"M3|{h}|{q}|{rep}"]
    w = adaptive_w(C2[f"npos|{h}|{q}|{rep}"][0])
    rU = pd.Series(U[te]).rank(pct=True).values; rM = pd.Series(m3).rank(pct=True).values
    return te, {"M3": m3, "U": U[te], "fixed_0.5": 0.5 * rU + 0.5 * rM, "adaptive": w * rU + (1 - w) * rM}

rows = []
for h in HOSPS:
    for q in FRACTIONS_C2:
        r = dict(hospital=h[:30], fraction=q,
                 cancers_train=np.mean([C2[f"npos|{h}|{q}|{k}"][0] for k in range(REPEATS_C2)]))
        for m in METHODS:
            r[m] = np.mean([pauc_80(y[c2_scores(h, q, k)[0]], c2_scores(h, q, k)[1][m]) for k in range(REPEATS_C2)])
        rows.append(r)
c2 = pd.DataFrame(rows)
c2_macro = c2.groupby("fraction")[["cancers_train"] + METHODS].mean()
c2_macro["best_of_M3_U"] = c2_macro[["M3", "U"]].max(1); c2_macro["adaptive_minus_best"] = c2_macro.adaptive - c2_macro.best_of_M3_U
print("Macro over", len(HOSPS), "hospitals:"); print(c2_macro.round(4).to_string())

In [ ]:
t0 = time.time(); meta = []
for base in ["M3", "U"]:
    for q in FRACTIONS_C2:
        est, var = [], []
        for h in HOSPS:
            te, _ = c2_scores(h, q, 0)
            bb = PatientBootstrap(pid[te], n_boot=N_BOOT, seed=SEED_C)
            sc0 = c2_scores(h, q, 0)[1]; ds = []
            for i in bb.samples:
                yi = y[te][i]
                if yi.sum() == 0: continue
                ds.append(pauc_80(yi, sc0["adaptive"][i]) - pauc_80(yi, sc0[base][i]))
            est.append(np.mean([pauc_80(y[te], c2_scores(h, q, k)[1]["adaptive"]) - pauc_80(y[te], c2_scores(h, q, k)[1][base])
                                for k in range(REPEATS_C2)]))
            var.append(np.var(ds) + 1e-8)
        mu, lo, hi, p, I2 = dersimonian_laird(est, var)
        meta.append(dict(comparison=f"adaptive vs {base}", fraction=q, pooled_delta=round(mu, 4),
                         ci95=f"[{lo:+.4f}, {hi:+.4f}]", ci_hi=hi, p_better=round(p, 4), I2=round(I2, 2),
                         hospitals_positive=f"{int((np.array(est) > 0).sum())}/{len(est)}"))
c2_meta = pd.DataFrame(meta)
for c in c2_meta.comparison.unique():
    m_ = c2_meta.comparison == c; c2_meta.loc[m_, "p_holm"] = holm(c2_meta.loc[m_, "p_better"].values).round(4)
c2_meta["sig_worse"] = c2_meta.ci_hi < 0; c2_meta = c2_meta.drop(columns="ci_hi")
print(c2_meta.to_string(index=False)); print(f"({time.time()-t0:.0f}s)")

## 7. Figure + save

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
st = {"M3": ("trained M3", "#d62728", "D", "-"), "U": ("zero-label U", "#9467bd", None, "--"),
      "fixed_0.5": ("fixed blend 0.5", "#ff7f0e", "o", ":"), "adaptive": (f"adaptive blend (k={K_PRIOR})", "#2ca02c", "s", "-")}
for ax, d, xl, tt in [(axes[0], c1.set_index("fraction"), "share of training patients labelled", "C1: within-dataset (fresh seeds)"),
                      (axes[1], c2_macro, "share of source-hospital patients labelled", "C2: unseen hospital (macro over 6)")]:
    for m, (lab, colr, mk, ls) in st.items():
        ax.plot(d.index * 100, d[m], label=lab, color=colr, marker=mk, ls=ls)
    ax.set_xscale("log"); ticks = [x * 100 for x in d.index]; ax.set_xticks(ticks); ax.set_xticklabels([f"{t:g}%" for t in ticks])
    ax.set_xlabel(xl); ax.set_ylabel("pAUC above 80% TPR"); ax.set_title(tt); ax.grid(alpha=.3)
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{WORK}/fig_adaptive_blend.png", dpi=200); plt.show()

c1.to_csv(f"{WORK}/results_step5c_c1.csv", index=False); c1_tests.to_csv(f"{WORK}/tests_step5c_c1.csv", index=False)
c2.to_csv(f"{WORK}/results_step5c_c2_per_hospital.csv", index=False); c2_macro.to_csv(f"{WORK}/results_step5c_c2_macro.csv")
c2_meta.to_csv(f"{WORK}/tests_step5c_c2_meta.csv", index=False)
print("saved")